# Portfolio default simulation

**Start here:** This deep dive expands on `07_advanced_quant/correlation_and_credit_models.ipynb`; read the overview first for the common copula, latent-factor, and recovery vocabulary.

From **name-level PDs** to a **systematic factor**, a **copula**, **correlated default indicators**, and a **portfolio loss distribution** — with a Gaussian vs Student-$t$ tail comparison.


## Concept

In a **one-factor** portfolio model, a single market draw $Z$ shifts all obligors’ **conditional** default probabilities. Names load on that factor through **asset correlation** $\rho$ and a **copula** (Gaussian or Student-$t$) that maps factor realizations to uniform margins.

The simulation loop is: draw a **systematic factor** (standard normal $Z$ for the Gaussian copula, or $M = Z/\sqrt{W/\nu}$ with $W\sim\chi^2(\nu)$ for the Student-$t$ copula); for each name compute $P(\text{default} \mid \cdot)$ via `conditional_default_prob` with threshold $c_i = \Phi^{-1}(\mathrm{PD}_i)$ or $c_i = t_\nu^{-1}(\mathrm{PD}_i)$ respectively; Bernoulli-draw defaults; aggregate **loss** $= \sum \mathbf{1}_{\text{default},i} \cdot N_i \cdot \mathrm{LGD}_i$. Repeating yields a **loss distribution**; we summarize **mean**, **VaR(99%)**, and **ES(99%)**.

Student-$t$ copulas exhibit **tail dependence**: conditional default probabilities react differently in joint stress than under a Gaussian copula, which often shows **fatter portfolio loss tails**.

## API walkthrough

Key pieces from `finstack_quant.valuations.correlation`:

- **`CreditExposure`** — name-level notional, PD, LGD, and systematic factor loadings $\beta$ with $\sum \beta_k^2 \le 1$.
- **`PortfolioLossConfig`** — path count, deterministic seed, confidence, and a Gaussian or Student-$t$ `CopulaSpec`.
- **`simulate_portfolio_loss`** — Rust-owned threshold construction, path-indexed Philox simulation, loss aggregation, nearest-rank loss VaR, and expected shortfall.
- **`CorrelatedBernoulli`**, **`joint_probabilities`**, **`correlation_bounds`**, and matrix helpers remain useful diagnostics.

Each path index owns a stable Philox substream, so the same seed produces identical losses under serial and parallel execution. The result uses positive losses: larger values are worse.

In [1]:
from finstack_quant.valuations.correlation import (
    CopulaSpec,
    CorrelatedBernoulli,
    CreditExposure,
    PortfolioLossConfig,
    RecoverySpec,
    cholesky_decompose,
    correlation_bounds,
    joint_probabilities,
    simulate_portfolio_loss,
    validate_correlation_matrix,
)
from finstack_quant.core.math.special_functions import (
    standard_normal_inv_cdf,
    student_t_inv_cdf,
)

print("Imports OK: canonical portfolio loss simulation plus correlation diagnostics")

Imports OK: canonical portfolio loss simulation plus correlation diagnostics


Fréchet–Hoeffding **correlation bounds** for two Bernoulli marginals, a tiny **2-name** joint view, and a **3×3** correlation matrix check plus its Cholesky factor.

In [2]:
lo, hi = correlation_bounds(0.02, 0.05)
print(f"Feasible correlation range for PDs (0.02, 0.05): [{lo:.4f}, {hi:.4f}]")

p11, p10, p01, p00 = joint_probabilities(0.02, 0.05, 0.15)
print(f"Joint probs (p11,p10,p01,p00) at rho=0.15: ({p11:.6f}, {p10:.6f}, {p01:.6f}, {p00:.6f})")
print(f"Sum check: {p11 + p10 + p01 + p00:.10f}")

cb = CorrelatedBernoulli(0.02, 0.05, 0.15)
print(f"CorrelatedBernoulli: {cb}")
print(f"P(both default): {cb.joint_p11:.6f}")

flat_3 = [1.0, 0.2, 0.1, 0.2, 1.0, 0.15, 0.1, 0.15, 1.0]
validate_correlation_matrix(flat_3, 3)
print("validate_correlation_matrix: 3x3 PSD example accepted")
L = cholesky_decompose(flat_3, 3)
print(f"Cholesky (flat, len={len(L)}): first 3 entries = {L[:3]}")

Feasible correlation range for PDs (0.02, 0.05): [-0.0328, 0.6227]
Joint probs (p11,p10,p01,p00) at rho=0.15: (0.005577, 0.014423, 0.044423, 0.935577)
Sum check: 1.0000000000
CorrelatedBernoulli: CorrelatedBernoulli(p1=0.0200, p2=0.0500, corr=0.1500)
P(both default): 0.005577
validate_correlation_matrix: 3x3 PSD example accepted
Cholesky (flat, len=9): first 3 entries = [0.99498743710662, 0.0, 0.1]


**Gaussian copula** — build once; thresholds $c_i = \Phi^{-1}(\mathrm{PD}_i)$; `tail_dependence` is zero. **Student-$t$** (e.g. 5 df) has **non-zero** tail dependence for the same $\rho$.

In [3]:
nu_t = 5.0
gauss = CopulaSpec.gaussian().build()
t_cop = CopulaSpec.student_t(nu_t).build()
rho_demo = 0.20
pd_demo = 0.03
c_gauss = standard_normal_inv_cdf(pd_demo)
c_t = student_t_inv_cdf(pd_demo, nu_t)
print(f"Gaussian model: {gauss.model_name}, tail_dep(0.5)={gauss.tail_dependence(0.5):.6f}")
print(f"Student-t model: {t_cop.model_name}, tail_dep(0.5)={t_cop.tail_dependence(0.5):.6f}")
print(
    f"Same Gaussian factor Z=-2.0 (Student-t W=1): Gauss cond PD={gauss.conditional_default_prob(c_gauss, [-2.0], rho_demo):.6f}, "
    f"t cond PD={t_cop.conditional_default_prob(c_t, [-2.0, 1.0], rho_demo):.6f}"
)

_ = RecoverySpec.constant(0.40)
print("RecoverySpec.constant(0.40) available for follow-on notebooks (no effect here)")

Gaussian model: One-Factor Gaussian Copula, tail_dep(0.5)=0.000000
Student-t model: Student-t Copula, tail_dep(0.5)=0.207031
Same Gaussian factor Z=-2.0 (Student-t W=1): Gauss cond PD=0.135059, t cond PD=0.043873
RecoverySpec.constant(0.40) available for follow-on notebooks (no effect here)


## Examples

**10-name** portfolio: individual PDs, **LGD 60%**, equal notionals **10M**, and one systematic loading $\beta=\sqrt{0.20}$ so asset correlation is **0.20**. Run **10,000** deterministic Philox paths with seed **42**. Compare Gaussian and Student-$t$ (5 df) results using Rust-owned **expected loss**, nearest-rank **VaR(99%)**, and **ES(99%)**.

In [4]:
pds = [0.02, 0.03, 0.05, 0.02, 0.04, 0.03, 0.06, 0.01, 0.03, 0.05]
rho = 0.20
n_sims = 10_000
exposures = [
    CreditExposure(
        id=f"name_{index + 1}",
        notional=10_000_000.0,
        default_probability=pd,
        lgd=0.60,
        factor_loadings=[rho**0.5],
    )
    for index, pd in enumerate(pds)
]

result_g = simulate_portfolio_loss(
    exposures,
    PortfolioLossConfig(n_sims, 42, 0.99, CopulaSpec.gaussian()),
)
result_t = simulate_portfolio_loss(
    exposures,
    PortfolioLossConfig(n_sims, 42, 0.99, CopulaSpec.student_t(nu_t)),
)

print(f"Simulations: {n_sims}, seed=42 (path-indexed Philox streams)")
print(
    f"Gaussian  — mean loss: {result_g.expected_loss:,.0f}, "
    f"VaR99: {result_g.var:,.0f}, ES99: {result_g.expected_shortfall:,.0f}"
)
print(
    f"Student-t — mean loss: {result_t.expected_loss:,.0f}, "
    f"VaR99: {result_t.var:,.0f}, ES99: {result_t.expected_shortfall:,.0f}"
)
print(
    f"Tail uplift (t vs Gauss): VaR99 {((result_t.var / result_g.var) - 1.0) * 100:+.2f}%, "
    f"ES99 {((result_t.expected_shortfall / result_g.expected_shortfall) - 1.0) * 100:+.2f}%"
)

Simulations: 10000, seed=42 (path-indexed Philox streams)
Gaussian  — mean loss: 2,048,400, VaR99: 18,000,000, ES99: 21,623,762
Student-t — mean loss: 2,083,200, VaR99: 24,000,000, ES99: 32,079,208
Tail uplift (t vs Gauss): VaR99 +33.33%, ES99 +48.35%


## Takeaways

- **`simulate_portfolio_loss`** owns copula-specific thresholds, random draws, default tests, loss aggregation, and tail statistics in Rust.
- **Factor loadings** satisfy $\sum \beta_k^2 \le 1$; one loading $\sqrt{\rho}$ reproduces one-factor asset correlation $\rho$.
- **Gaussian** vs **Student-$t$** copulas can match similar unconditional behavior but differ in joint stress and tail loss.
- **VaR / ES are loss-positive**: larger values are worse, VaR uses the nearest-rank confidence quantile, and ES includes the VaR observation.
- Path-indexed Philox streams preserve deterministic seeds and exact serial/parallel equality.